# Nettoyage et préparation des données — Accidents corporels 2024

**Projet Data Science L3 — Université Catholique de Lille**

**Auteur du nettoyage** : Enzo — **Binôme modélisation** : Issam, Noé

---

## Objectif du notebook

Produire à partir du fichier brut `accidents_2024.csv` trois datasets prêts à la modélisation :

- `X_lab_encoded.csv` — features encodées ordinalement (pour XGBoost, Random Forest)
- `X_oh_encoding.csv` — features en One-Hot Encoding (pour Régression Logistique)
- `y_gravite.csv` — variable cible encodée en entiers 0/1/2

Plus un artefact : `label_encoding_mappings.json` — correspondance `code ↔ modalité` pour toutes les colonnes ordinalement encodées (indispensable pour interpréter les feature importances).

## Principes méthodologiques

1. **Aucune imputation n'est réalisée dans ce notebook.** Les valeurs manquantes sont conservées dans `X`. Chaque binôme appliquera sa propre stratégie d'imputation dans un `Pipeline` sklearn **après** le `train_test_split`, pour éviter toute fuite d'information entre train et test.
2. **Les doublons sont conservés.** Justification détaillée dans la section dédiée.
3. **L'encodage ordinal respecte la sémantique** quand elle existe (luminosité, état de la surface), pour donner du sens aux splits des modèles à base d'arbres et faciliter l'interprétation.


## 1. Chargement des données


### 1.1 Encoding du fichier source

Le CSV d'origine est produit par un export Excel en encodage **Windows-1252 (cp1252)**. Utiliser `latin-1` fonctionne partiellement mais introduit un bug sur les apostrophes typographiques : `l'accident` devient `l\x92accident`, car `\x92` est valide en latin-1 mais représente l'apostrophe typographique en cp1252. On utilise donc directement `cp1252`.


In [1]:
import os
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

In [2]:
path = "donnees/"
accidents = pd.read_csv(
    path + "accidents_2024.csv",
    sep=";",
    encoding="cp1252",
    low_memory=False,
)
donnees_brute = accidents.copy()
print(f"Shape brut : {accidents.shape}")

Shape brut : (54402, 14)


In [3]:
accidents.head(5)

,Num_Acc,Département,Agglomération,Luminosité,Météo (conditions atmos.),Type de collision,Catégorie de route,Régime de circulation,Nombre de voies,État de la surface,Infrastructure,Situation de l’accident,Vitesse max autorisée,Gravité (label)
0,"2,02E+11",70,Hors agglomération,Crépuscule / aube,Brouillard / fumée,2 véhicules - frontale,Route départementale,Bidirectionnelle,2 voie(s),Normale,Aucun,Sur chaussée,90 km/h,Blessé hospitalisé
1,"2,02E+11",21,En agglomération,Plein jour,Temps éblouissant,Autre collision,Voie communale,Bidirectionnelle,2 voie(s),Autre,Aucun,Sur chaussée,30 km/h,Blessé hospitalisé
2,"2,02E+11",15,Hors agglomération,Crépuscule / aube,Normale,Autre collision,Voie communale,Bidirectionnelle,2 voie(s),Normale,Aucun,Sur accotement,50 km/h,Blessé hospitalisé
3,"2,02E+11",14,En agglomération,Plein jour,Temps éblouissant,2 véhicules - par le côté,Voie communale,Bidirectionnelle,4 voie(s),Normale,Autres,Sur chaussée,50 km/h,Blessé léger
4,"2,02E+11",13,Hors agglomération,Nuit avec éclairage public allumé,Pluie légère,3 véhicules et + - collisions multiples,Autoroute,Bidirectionnelle,4 voie(s),Mouillée,Aucun,Sur chaussée,50 km/h,Blessé léger


In [4]:
accidents.info()

<class 'pandas.DataFrame'>
RangeIndex: 54402 entries, 0 to 54401
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Num_Acc                    54402 non-null  str  
 1   Département                54402 non-null  str  
 2   Agglomération              54402 non-null  str  
 3   Luminosité                 54402 non-null  str  
 4   Météo (conditions atmos.)  54391 non-null  str  
 5   Type de collision          54401 non-null  str  
 6   Catégorie de route         54402 non-null  str  
 7   Régime de circulation      54402 non-null  str  
 8   Nombre de voies            54402 non-null  str  
 9   État de la surface         54402 non-null  str  
 10  Infrastructure             54402 non-null  str  
 11  Situation de l’accident    54402 non-null  str  
 12  Vitesse max autorisée      54402 non-null  str  
 13  Gravité (label)            54402 non-null  str  
dtypes: str(14)
memory usage: 5.8 MB


## 2. Exploration initiale

On regarde : la distribution de la cible, les valeurs manquantes, et les modalités uniques pour repérer les champs textuels à nettoyer.


In [5]:
accidents.describe(include="all")

,Num_Acc,Département,Agglomération,Luminosité,Météo (conditions atmos.),Type de collision,Catégorie de route,Régime de circulation,Nombre de voies,État de la surface,Infrastructure,Situation de l’accident,Vitesse max autorisée,Gravité (label)
count,54402,54402,54402,54402,54391,54401,54402,54402,54402,54402,54402,54402,54402,54402
unique,1,107,2,5,9,8,8,5,14,10,11,7,35,3
top,"2,02E+11",75,En agglomération,Plein jour,Normale,2 véhicules - par le côté,Voie communale,Bidirectionnelle,2 voie(s),Normale,Aucun,Sur chaussée,50 km/h,Blessé léger
freq,54402,4191,34010,35580,41791,16371,22910,33576,35076,42418,45698,44782,24497,34744


In [6]:
print("Valeurs manquantes par colonne :")
print(accidents.isnull().sum())

Valeurs manquantes par colonne :
Num_Acc                       0
Département                   0
Agglomération                 0
Luminosité                    0
Météo (conditions atmos.)    11
Type de collision             1
Catégorie de route            0
Régime de circulation         0
Nombre de voies               0
État de la surface            0
Infrastructure                0
Situation de l’accident       0
Vitesse max autorisée         0
Gravité (label)               0
dtype: int64


In [7]:
print("Modalités uniques par colonne :")
print(accidents.nunique())

Modalités uniques par colonne :
Num_Acc                        1
Département                  107
Agglomération                  2
Luminosité                     5
Météo (conditions atmos.)      9
Type de collision              8
Catégorie de route             8
Régime de circulation          5
Nombre de voies               14
État de la surface            10
Infrastructure                11
Situation de l’accident        7
Vitesse max autorisée         35
Gravité (label)                3
dtype: int64


In [8]:
print("Distribution de la cible :")
print(accidents["Gravité (label)"].value_counts())
print()
print("Proportions :")
print(accidents["Gravité (label)"].value_counts(normalize=True).round(3))

Distribution de la cible :
Gravité (label)
Blessé léger          34744
Blessé hospitalisé    16432
Tué                    3226
Name: count, dtype: int64

Proportions :
Gravité (label)
Blessé léger          0.639
Blessé hospitalisé    0.302
Tué                   0.059
Name: proportion, dtype: float64


**Observations clés :**

- **54 402 accidents**, 14 colonnes dont 13 catégorielles au format texte et 1 identifiant.
- **Cible fortement déséquilibrée** : 63,9 % de blessés légers, 30,2 % d'hospitalisés, 5,9 % de tués. Ratio max/min ≈ 11x. À gérer côté modélisation avec `class_weight='balanced'` (LogReg, RF) ou `sample_weight` / `scale_pos_weight` (XGBoost).
- **Valeurs manquantes très peu nombreuses** : 11 sur Météo, 1 sur Type de collision. Les autres « manquants » sont encodés en clair comme `"Non renseigné"` ou `"#VALEURMULTI"` et ne sont pas détectés par `isnull()`.
- **107 départements** distincts : une OHE directe génèrerait trop de colonnes → regroupement en régions nécessaire (section 10).
- **35 modalités de vitesse** : la colonne est en format texte (ex. `"50 km/h"`), il faudra extraire la valeur numérique.


## 3. Traitement de `Num_Acc`


In [9]:
print(f"Nombre de Num_Acc uniques : {accidents['Num_Acc'].nunique()}")
print(f"Échantillon : {accidents['Num_Acc'].head(3).tolist()}")

Nombre de Num_Acc uniques : 1
Échantillon : ['2,02E+11', '2,02E+11', '2,02E+11']


### 3.1 L'identifiant est corrompu

Toutes les valeurs de `Num_Acc` sont identiques à `"2,02E+11"` — c'est la notation scientifique qu'Excel a appliquée au moment de l'export (l'identifiant original, probablement un entier à 12 chiffres du type `202400000123`, a été converti en `2.02E+11` avant l'export CSV).

**Conséquence importante** : on ne peut pas utiliser `Num_Acc` pour regrouper les lignes appartenant au même accident. Cela a des implications fortes sur le traitement des doublons (section 4).

On supprime la colonne, qui n'apporte aucune information prédictive.


In [10]:
accidents = accidents.drop(columns=["Num_Acc"])
print(f"Shape après suppression : {accidents.shape}")

Shape après suppression : (54402, 13)


## 4. Analyse des doublons

C'est la décision la plus délicate de ce nettoyage. On l'analyse quantitativement avant de trancher.


In [11]:
dup_mask = accidents.duplicated(keep=False)
n_dup_complets = accidents.duplicated().sum()
n_lignes_impliquees = dup_mask.sum()

print(f"Doublons complets : {n_dup_complets} ({n_dup_complets/len(accidents)*100:.1f}%)")
print(f"Lignes impliquées dans un groupe doublon : {n_lignes_impliquees} ({n_lignes_impliquees/len(accidents)*100:.1f}%)")

Doublons complets : 13221 (24.3%)
Lignes impliquées dans un groupe doublon : 18372 (33.8%)


In [12]:
dist_dup = accidents.loc[dup_mask, "Gravité (label)"].value_counts(normalize=True).round(3)
dist_all = accidents["Gravité (label)"].value_counts(normalize=True).round(3)

comparaison = pd.DataFrame({
    "Dans les doublons": dist_dup,
    "Dataset global": dist_all,
})
print(comparaison)

                    Dans les doublons  Dataset global
Gravité (label)                                      
Blessé léger                    0.742           0.639
Blessé hospitalisé              0.236           0.302
Tué                             0.022           0.059


### 4.1 Interprétation des chiffres

Un tiers du dataset (33,8 %) est impliqué dans un groupe de doublons, et la distribution de gravité dans ces doublons est **significativement biaisée vers les classes légères** : 74 % de blessés légers contre 64 % dans le dataset global ; 2 % de tués contre 6 %.

### 4.2 Hypothèse sur l'origine des doublons

La base BAAC (source ONISR) contient à l'origine **une ligne par usager impliqué** dans un accident. Chaque accident multi-usagers génère donc plusieurs lignes partageant exactement les mêmes caractéristiques contextuelles (météo, luminosité, type de collision, département…) mais distinguées par l'identité et la gravité individuelle de chaque usager.

Le biais observé est cohérent avec cette hypothèse : un accident mortel produit typiquement un mort + des blessés de gravités variées (hospitalisés, légers), donc les lignes ne sont **pas** identiques entre elles. À l'inverse, un accident léger impliquant plusieurs passagers produit souvent plusieurs lignes rigoureusement identiques (même contexte + même gravité légère pour tous).

### 4.3 Décision : conserver les doublons

**Chaque ligne représente un usager réel distinct.** Les supprimer reviendrait à jeter de vraies observations et à introduire un biais en faveur des accidents multi-gravités (accidents graves). Notre tâche prédictive est donc formulée au niveau usager : *« étant donné le contexte d'un accident, quelle gravité un usager impliqué subira-t-il ? »*

### 4.4 Limite à reconnaître dans le rapport

Puisque `Num_Acc` est corrompu, on ne peut pas grouper les usagers d'un même accident. Le `train_test_split` stratifié sur `y` risque donc de placer des usagers du même accident dans des splits différents (fuite légère, car les features contextuelles seront identiques de part et d'autre). Un **`GroupShuffleSplit` basé sur `Num_Acc`** aurait été plus rigoureux, mais impossible ici.


## 5. Traitement de la colonne `Type de collision`

Une seule ligne est en NaN véritable sur cette colonne (sur 54 402). Vu le volume négligeable et l'absence d'une modalité sémantique évidente à lui assigner, on la supprime.


In [13]:
print(f"NaN avant : {accidents['Type de collision'].isna().sum()}")
accidents = accidents.dropna(subset=["Type de collision"])
print(f"NaN après : {accidents['Type de collision'].isna().sum()}")
print(f"Shape : {accidents.shape}")

NaN avant : 1
NaN après : 0
Shape : (54401, 13)


## 6. Extraction numérique de `Vitesse max autorisée`

La colonne est au format texte `"50 km/h"`, avec quelques cas non numériques (`"Non renseigné"`, `"#VALEURMULTI km/h"`). On extrait la valeur numérique par regex.


In [14]:
print("Distribution brute (extrait) :")
print(accidents["Vitesse max autorisée"].value_counts().head(10))

Distribution brute (extrait) :
Vitesse max autorisée
50 km/h          24497
30 km/h           9373
80 km/h           7217
90 km/h           5054
70 km/h           3879
110 km/h          1788
130 km/h          1007
Non renseigné      740
20 km/h            233
10 km/h            152
Name: count, dtype: int64


In [15]:
accidents["Vitesse_num"] = accidents["Vitesse max autorisée"].str.extract(r"(\d+)")
accidents["Vitesse_num"] = pd.to_numeric(accidents["Vitesse_num"])
print(accidents["Vitesse_num"].describe())

count    53661.000000
mean        59.072119
std         25.244885
min          1.000000
25%         50.000000
50%         50.000000
75%         80.000000
max        900.000000
Name: Vitesse_num, dtype: float64


### 6.1 Valeurs aberrantes

La distribution montre un maximum à **900 km/h** : aberrant. On observe des valeurs à 140, 300, 500, 700, 800, 900. Le seuil réglementaire maximal en France est **130 km/h** (autoroutes). On considère donc toute vitesse > 130 comme une erreur de saisie et on la passe en NaN (l'imputation sera faite par chaque binôme dans son pipeline).


In [16]:
valeurs_aberrantes = accidents["Vitesse_num"] > 130
print(f"Nombre de valeurs aberrantes (> 130 km/h) : {valeurs_aberrantes.sum()}")
print("Valeurs concernées :")
print(accidents.loc[valeurs_aberrantes, "Vitesse_num"].value_counts())

Nombre de valeurs aberrantes (> 130 km/h) : 20
Valeurs concernées :
Vitesse_num
500.0    12
300.0     2
900.0     2
800.0     1
140.0     1
700.0     1
301.0     1
Name: count, dtype: int64


In [17]:
accidents.loc[valeurs_aberrantes, "Vitesse_num"] = np.nan
print(accidents["Vitesse_num"].describe())

count    53641.000000
mean        58.901363
std         23.374811
min          1.000000
25%         50.000000
50%         50.000000
75%         80.000000
max        130.000000
Name: Vitesse_num, dtype: float64


### 6.2 Remarque : vitesses très basses

On observe des vitesses à 1, 2, 5 km/h (224 lignes entre 1 et 10 km/h). Ces valeurs ne correspondent à aucune limitation réglementaire française standard. Elles proviennent probablement de zones à très faible trafic (parkings, voies piétonnes). On les conserve telles quelles : elles sont rares (<0,5 %), plausibles comme zones spécifiques, et les retirer reviendrait à prendre une décision éditoriale difficile à justifier.


### 6.3 NaN conservés pour l'aval

Les NaN de `Vitesse_num` (aberrantes + `Non renseigné` + `#VALEURMULTI` transformés par la regex) **ne sont pas imputés ici**. Voir la section 16 « Stratégie d'imputation ».


In [18]:
print(f"NaN dans Vitesse_num (à traiter côté modélisation) : {accidents['Vitesse_num'].isna().sum()}")

NaN dans Vitesse_num (à traiter côté modélisation) : 760


## 7. Extraction numérique de `Nombre de voies`

Même logique : extraction par regex, les `Non renseigné` et `#VALEURMULTI voie(s)` deviennent NaN.


In [19]:
print("Distribution brute :")
print(accidents["Nombre de voies"].value_counts())

Distribution brute :
Nombre de voies
2 voie(s)               35076
1 voie(s)                5965
4 voie(s)                4754
3 voie(s)                4510
Non renseigné            2218
6 voie(s)                 959
5 voie(s)                 536
8 voie(s)                 223
7 voie(s)                  57
#VALEURMULTI voie(s)       46
10 voie(s)                 32
9 voie(s)                  14
12 voie(s)                 10
11 voie(s)                  1
Name: count, dtype: int64


In [20]:
accidents["Voies_num"] = accidents["Nombre de voies"].str.extract(r"(\d+)")
accidents["Voies_num"] = pd.to_numeric(accidents["Voies_num"])
print(accidents["Voies_num"].describe())
print(f"\nNaN : {accidents['Voies_num'].isna().sum()}")

count    52137.000000
mean         2.298886
std          1.057187
min          1.000000
25%          2.000000
50%          2.000000
75%          2.000000
max         12.000000
Name: Voies_num, dtype: float64

NaN : 2264


Les valeurs vont de 1 à 12 voies, ce qui reste plausible (autoroutes larges, échangeurs). Pas de valeurs aberrantes à traiter. Les NaN (~2 264 lignes, soit ~4 %) sont conservés pour imputation aval.


## 8. Suppression des colonnes texte désormais redondantes


In [21]:
accidents = accidents.drop(columns=["Vitesse max autorisée", "Nombre de voies"])
print(f"Shape : {accidents.shape}")
print(f"Colonnes : {accidents.columns.tolist()}")

Shape : (54401, 13)
Colonnes : ['Département', 'Agglomération', 'Luminosité', 'Météo (conditions atmos.)', 'Type de collision', 'Catégorie de route', 'Régime de circulation', 'État de la surface', 'Infrastructure', 'Situation de l’accident', 'Gravité (label)', 'Vitesse_num', 'Voies_num']


## 9. Uniformisation des valeurs manquantes catégorielles

Plusieurs colonnes catégorielles utilisent la chaîne `"Non renseigné"` pour signaler l'absence d'information. On la renomme en `"Inconnu"` pour la conserver comme une modalité à part entière. L'absence d'information peut elle-même être corrélée à la gravité (par exemple, un accident tellement grave que certains renseignements n'ont pas été collectés sur place).

**Choix méthodologique :** on traite `"Non renseigné"` comme une modalité informative, et **non** comme une valeur manquante à imputer. C'est une décision différente de celle prise pour les NaN véritables sur les colonnes numériques (Vitesse, Voies, Météo), qui seront imputés parce que sans signal catégoriel interprétable.

**Note :** dans le mapping final, les rares NaN véritables (colonne Météo notamment) sont également étiquetés `"Inconnu"` pour garantir la validité stricte du JSON produit. Sémantiquement, cette modalité regroupe donc à la fois les cas explicitement non renseignés par l'agent (`Non renseigné` d'origine) et les cas de donnée absente à la source (NaN).


In [22]:
print("Colonnes contenant 'Non renseigné' :")
for col in accidents.columns:
    if accidents[col].dtype.name in ("object", "str", "string"):
        n = (accidents[col] == "Non renseigné").sum()
        if n > 0:
            print(f"  {col}: {n}")

Colonnes contenant 'Non renseigné' :
  Type de collision: 6
  Régime de circulation: 2971
  État de la surface: 2
  Infrastructure: 448


In [23]:
accidents = accidents.replace("Non renseigné", "Inconnu")
print("Remplacement effectué.")

Remplacement effectué.


## 10. Regroupement Département → Région

**Motivation :** 107 départements × OHE = 107 colonnes supplémentaires, risque d'explosion dimensionnelle et de modalités à très faible effectif (ex. certains DOM-TOM avec 2 à 40 observations). On regroupe selon le découpage administratif actuel en 13 régions métropolitaines + 1 regroupement Outre-mer = **14 modalités**.

**Limite :** cette agrégation perd la granularité géographique fine. Un département rural et sa métropole voisine finissent dans la même région. C'est un compromis dimensionnalité / information qu'il faut mentionner dans le rapport.


In [24]:
region_to_depts = {
    'Île-de-France':            ['75', '77', '78', '91', '92', '93', '94', '95'],
    'Auvergne-Rhône-Alpes':     ['01', '03', '07', '15', '26', '38', '42', '43', '63', '69', '73', '74'],
    'Bretagne':                 ['22', '29', '35', '56'],
    'Bourgogne-Franche-Comté':  ['21', '25', '39', '58', '70', '71', '89', '90'],
    'Normandie':                ['14', '27', '50', '61', '76'],
    'Hauts-de-France':          ['02', '59', '60', '62', '80'],
    'Grand Est':                ['08', '10', '51', '52', '54', '55', '57', '67', '68', '88'],
    'Pays de la Loire':         ['44', '49', '53', '72', '85'],
    'Centre-Val de Loire':      ['18', '28', '36', '37', '41', '45'],
    'Nouvelle-Aquitaine':       ['16', '17', '19', '23', '24', '33', '40', '47', '64', '79', '86', '87'],
    'Occitanie':                ['09', '11', '12', '30', '31', '32', '34', '46', '48', '65', '66', '81', '82'],
    'PACA':                     ['04', '05', '06', '13', '83', '84'],
    'Corse':                    ['2A', '2B'],
    'Outre-mer':                ['971', '972', '973', '974', '975', '976', '977', '978', '986', '987', '988'],
}

# Aplatissement : {dept: region}
dept_to_region = {d: r for r, depts in region_to_depts.items() for d in depts}

In [25]:
# Normalisation des codes département sur 2 chiffres (préserve '2A' / '2B')
accidents["Département"] = accidents["Département"].str.zfill(2)
accidents["Région"] = accidents["Département"].map(dept_to_region)

print(f"Régions non mappées (codes inconnus) : {accidents['Région'].isna().sum()}")
print()
print("Distribution par région :")
print(accidents["Région"].value_counts())

Régions non mappées (codes inconnus) : 0

Distribution par région :
Région
Île-de-France              15365
Auvergne-Rhône-Alpes        5854
PACA                        4732
Nouvelle-Aquitaine          4272
Occitanie                   3794
Outre-mer                   3344
Grand Est                   3280
Hauts-de-France             2704
Normandie                   2374
Pays de la Loire            2360
Bretagne                    2340
Centre-Val de Loire         1719
Bourgogne-Franche-Comté     1691
Corse                        572
Name: count, dtype: int64


In [26]:
accidents = accidents.drop(columns=["Département"])
print(f"Shape : {accidents.shape}")

Shape : (54401, 13)


## 11. Séparation features / cible


In [27]:
X = accidents.drop(columns=["Gravité (label)"])
y = accidents["Gravité (label)"]
print(f"X : {X.shape}, y : {y.shape}")
print()
print("NaN par colonne dans X (à imputer aval) :")
print(X.isna().sum())

X : (54401, 12), y : (54401,)

NaN par colonne dans X (à imputer aval) :
Agglomération                   0
Luminosité                      0
Météo (conditions atmos.)      11
Type de collision               0
Catégorie de route              0
Régime de circulation           0
État de la surface              0
Infrastructure                  0
Situation de l’accident         0
Vitesse_num                   760
Voies_num                    2264
Région                          0
dtype: int64


## 12. Encodage de la cible

On utilise un mapping explicite plutôt qu'un `LabelEncoder` sur `y`, pour contrôler l'ordre et préserver la sémantique de gravité croissante (0 = le moins grave, 2 = le plus grave). Cet ordre facilitera l'interprétation des matrices de confusion.


In [28]:
gravite_mapping = {
    'Blessé léger':       0,
    'Blessé hospitalisé': 1,
    'Tué':                2,
}
y_encoded = y.map(gravite_mapping)
print(y_encoded.value_counts().sort_index())

Gravité (label)
0    34743
1    16432
2     3226
Name: count, dtype: int64


## 13. Encodage ordinal des features (pour XGBoost et Random Forest)

### 13.1 Choix : `OrdinalEncoder` plutôt que `LabelEncoder`

`sklearn.preprocessing.LabelEncoder` est conçu pour encoder la cible `y`, pas des features. Pour les features, `OrdinalEncoder` est l'outil approprié :

- il accepte directement un DataFrame multi-colonnes ;
- il permet de spécifier l'ordre des modalités via `categories=[...]` ;
- `handle_unknown='use_encoded_value'` permet de gérer proprement les modalités inconnues au moment de la prédiction (utile si on déploie le modèle sur un nouveau fichier).

### 13.2 Ordre sémantique pour Luminosité et État de la surface

Pour ces deux colonnes, un ordre naturel existe sur l'axe « gravité des conditions ». On l'impose explicitement :

- **Luminosité** : `Plein jour < Crépuscule/aube < Nuit avec éclairage allumé < Nuit avec éclairage non allumé < Nuit sans éclairage`. Intuition : plus la visibilité est mauvaise, plus le risque est élevé.
- **État de la surface** : `Normale < Mouillée < Flaques < Inondée < Enneigée < Boue < Verglacée < Corps gras < Autre < Inconnu`. Intuition : plus l'adhérence est faible, plus le risque est élevé. Les dernières modalités (Autre, Inconnu) sont placées en fin de liste faute de position sémantique claire.

**Remarque technique :** l'ordre imposé n'affecte pas directement la performance des modèles à base d'arbres (XGBoost, RF), qui trouvent des splits optimaux indépendamment de l'ordre. En revanche il facilite **grandement l'interprétation** des feature importances et des splits dans le rapport (par exemple : « le modèle sépare à surface_encoded ≥ 5, c'est-à-dire à partir de Boue »).

Pour les autres colonnes sans ordre sémantique clair (Région, Type de collision, Infrastructure…), on laisse l'ordre alphabétique par défaut.


In [29]:
luminosite_order = [
    'Plein jour',
    'Crépuscule / aube',
    'Nuit avec éclairage public allumé',
    'Nuit avec éclairage public non allumé',
    'Nuit sans éclairage public',
]

surface_order = [
    'Normale',
    'Mouillée',
    'Flaques',
    'Inondée',
    'Enneigée',
    'Boue',
    'Verglacée',
    'Corps gras / huile',
    'Autre',
    'Inconnu',
]

In [30]:
# Vérification : toutes les modalités présentes dans X sont-elles couvertes par l'ordre défini ?
for col, ordre in [("Luminosité", luminosite_order), ("État de la surface", surface_order)]:
    reelles = set(X[col].dropna().unique())
    declarees = set(ordre)
    manquantes = reelles - declarees
    superflues = declarees - reelles
    print(f"{col} — modalités réelles non déclarées  : {manquantes or '(aucune)'}")
    print(f"{col} — modalités déclarées non présentes : {superflues or '(aucune)'}")
    print()

Luminosité — modalités réelles non déclarées  : (aucune)
Luminosité — modalités déclarées non présentes : (aucune)

État de la surface — modalités réelles non déclarées  : (aucune)
État de la surface — modalités déclarées non présentes : (aucune)



In [31]:
X_lab_encoded = X.copy()

ordered_cols = {
    "Luminosité":         luminosite_order,
    "État de la surface": surface_order,
}

mappings = {}  # {colonne: {code: modalité}}

for col, categories in ordered_cols.items():
    enc = OrdinalEncoder(
        categories=[categories],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )
    X_lab_encoded[[col]] = enc.fit_transform(X_lab_encoded[[col]])
    mappings[col] = {i: v for i, v in enumerate(categories)}

In [32]:
# Colonnes catégorielles restantes (ordre alphabétique par défaut)
remaining_cat_cols = [
    c for c in X_lab_encoded.columns
    if X_lab_encoded[c].dtype.name in ("object", "str", "string")
]
print(f"Colonnes encodées sans ordre sémantique : {remaining_cat_cols}")

enc_default = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)
X_lab_encoded[remaining_cat_cols] = enc_default.fit_transform(X_lab_encoded[remaining_cat_cols])

# On complète les mappings pour ces colonnes
for col, cats in zip(remaining_cat_cols, enc_default.categories_):
    mappings[col] = {
        i: ("Inconnu" if pd.isna(v) else v)
        for i, v in enumerate(cats)
    }

Colonnes encodées sans ordre sémantique : ['Agglomération', 'Météo (conditions atmos.)', 'Type de collision', 'Catégorie de route', 'Régime de circulation', 'Infrastructure', 'Situation de l’accident', 'Région']


In [33]:
print("Aperçu :")
print(X_lab_encoded.head())
print()
print("Dtypes :")
print(X_lab_encoded.dtypes)

Aperçu :
   Agglomération  Luminosité  Météo (conditions atmos.)  Type de collision  \
0            1.0         1.0                        1.0                0.0   
1            0.0         0.0                        7.0                5.0   
2            1.0         1.0                        3.0                5.0   
3            0.0         0.0                        7.0                1.0   
4            1.0         2.0                        5.0                3.0   

   Catégorie de route  Régime de circulation  État de la surface  \
0                 5.0                    0.0                 0.0   
1                 7.0                    0.0                 8.0   
2                 7.0                    0.0                 0.0   
3                 7.0                    0.0                 0.0   
4                 0.0                    0.0                 1.0   

   Infrastructure  Situation de l’accident  Vitesse_num  Voies_num  Région  
0             0.0                   

### 13.3 Sauvegarde des mappings

Sans ces mappings, il serait impossible d'interpréter les feature importances des modèles. Par exemple, si XGBoost retourne « la modalité 4 de `Type de collision` est la plus discriminante », on ne saurait pas à quelle modalité réelle cela correspond. On sérialise donc les mappings en JSON à côté des CSV.


In [34]:
print("Exemples de mappings :")
print(f"Luminosité       : {mappings['Luminosité']}")
print(f"Type de collision : {mappings['Type de collision']}")

Exemples de mappings :
Luminosité       : {0: 'Plein jour', 1: 'Crépuscule / aube', 2: 'Nuit avec éclairage public allumé', 3: 'Nuit avec éclairage public non allumé', 4: 'Nuit sans éclairage public'}
Type de collision : {0: '2 véhicules - frontale', 1: '2 véhicules - par le côté', 2: '2 véhicules - par l’arrière', 3: '3 véhicules et + - collisions multiples', 4: '3 véhicules et + - en chaîne', 5: 'Autre collision', 6: 'Inconnu', 7: 'Sans collision'}


## 14. One-Hot Encoding (pour la Régression Logistique)

Pour un modèle linéaire comme la régression logistique, encoder une variable catégorielle par un entier imposerait une relation d'ordre arbitraire entre les modalités (coefficient unique multiplicateur). Le One-Hot Encoding règle ce problème en créant une indicatrice par modalité.

**Remarques pour l'équipe Régression Logistique :**

- On utilise `dtype=float` pour éviter les dtypes booléens qui peuvent poser problème selon la version de scikit-learn.
- `drop_first=False` par défaut. **À discuter côté modélisation** : dropper la première modalité (`drop_first=True`) évite la colinéarité parfaite entre les indicatrices et l'intercept. Avec régularisation (`penalty='l2'`, valeur par défaut), ce n'est pas strictement nécessaire, mais c'est plus propre et cela accélère la convergence.


In [35]:
X_oh_encoding = X.copy()
cat_cols_oh = [
    c for c in X_oh_encoding.columns
    if X_oh_encoding[c].dtype.name in ("object", "str", "string")
]
X_oh_encoding = pd.get_dummies(X_oh_encoding, columns=cat_cols_oh, dtype=float)
print(f"Shape : {X_oh_encoding.shape}")
print(f"Premières colonnes : {X_oh_encoding.columns.tolist()[:6]}")

Shape : (54401, 81)
Premières colonnes : ['Vitesse_num', 'Voies_num', 'Agglomération_En agglomération', 'Agglomération_Hors agglomération', 'Luminosité_Crépuscule / aube', 'Luminosité_Nuit avec éclairage public allumé']


## 15. Sauvegarde des datasets


In [36]:
os.makedirs("donnees/clean", exist_ok=True)

X_lab_encoded.to_csv("donnees/clean/X_lab_encoded.csv", index=False)
X_oh_encoding.to_csv("donnees/clean/X_oh_encoding.csv", index=False)
y_encoded.to_csv("donnees/clean/y_gravite.csv", index=False)

with open("donnees/clean/label_encoding_mappings.json", "w", encoding="utf-8") as f:
    json.dump(mappings, f, ensure_ascii=False, indent=2, allow_nan=False)

print("Fichiers écrits :")
for f in sorted(os.listdir("donnees/clean")):
    print(f"  - {f}")

Fichiers écrits :
  - X_lab_encoded.csv
  - X_oh_encoding.csv
  - label_encoding_mappings.json
  - y_gravite.csv


## 16. Stratégie d'imputation — guide pour chaque modèle

**Aucune imputation n'a été réalisée dans ce notebook.** Les NaN restants dans `X` sont :

| Colonne | NaN | Nature |
|---|---|---|
| `Météo (conditions atmos.)` | 11 | NaN véritables à la source |
| `Vitesse_num` | ~760 | Aberrantes > 130 km/h + `Non renseigné` + `#VALEURMULTI` |
| `Voies_num` | ~2 264 | `Non renseigné` + `#VALEURMULTI` |

### 16.1 Pourquoi pas d'imputation ici

Imputer avant le `train_test_split` crée une **fuite de données** : la médiane ou le mode calculés sur l'ensemble complet intègrent de l'information du futur test set dans le train. Même si l'impact numérique est faible vu les proportions, c'est une erreur méthodologique qu'un jury relèvera (règle symétrique à celle du `StandardScaler`).

Chaque binôme doit donc inclure l'imputation dans son `Pipeline` sklearn, après le split.

### 16.2 Stratégie par modèle

**Régression Logistique** — Fichier : `X_oh_encoding.csv`

`LogisticRegression` ne supporte pas les NaN. Imputation **obligatoire** avant fit. Le OHE ne contient pas de NaN dans les indicatrices catégorielles (`get_dummies` ne crée pas d'indicatrice NaN par défaut, ce qui signifie que les 11 lignes de Météo manquante ont toutes leurs indicatrices Météo à 0 — comportement acceptable ici). Il reste donc uniquement les NaN sur `Vitesse_num` et `Voies_num` à imputer.

```python
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(class_weight='balanced', max_iter=1000)),
])
pipe.fit(X_train, y_train)
```

**Random Forest** — Fichier : `X_lab_encoded.csv`

`RandomForestClassifier` ne supporte pas les NaN non plus. Imputation **obligatoire** avant fit (médiane pour les colonnes numériques ; mode pour les colonnes catégorielles si on décidait finalement d'imputer Météo). Pas besoin de scaler. `class_weight='balanced'` pour le déséquilibre.

```python
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf',     RandomForestClassifier(class_weight='balanced', random_state=42)),
])
```

**XGBoost** — Fichier : `X_lab_encoded.csv`

XGBoost **gère nativement les valeurs manquantes** : pour chaque split d'arbre, il apprend dans quelle branche envoyer les NaN afin de minimiser la perte. C'est une des forces majeures du modèle.

**Implication :** ne pas imputer est un choix défendable et probablement optimal. Imputer reviendrait à masquer le signal « information manquante » que XGBoost peut exploiter. À tester en comparaison dans le rapport.

```python
# Sans imputer — laisse XGBoost gérer les NaN
model = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    # scale_pos_weight / sample_weight à calibrer pour le déséquilibre
    random_state=42,
)
model.fit(X_train, y_train)  # X_train peut contenir des NaN
```

### 16.3 Protocole commun à tous les modèles

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
```

- `stratify=y` pour conserver les proportions des trois classes dans train et test.
- `random_state=42` pour la reproductibilité entre les trois modèles (important pour que les comparaisons soient loyales).


## 17. Résumé des décisions de nettoyage

| Décision | Choix | Justification |
|---|---|---|
| Encoding fichier | `cp1252` | Corrige l'apostrophe typographique |
| `Num_Acc` | Supprimée | Identifiant corrompu (tous à `2,02E+11`) |
| Doublons | Conservés | Une ligne = un usager réel (hypothèse BAAC multi-usagers) |
| 1 NaN sur Type de collision | Ligne supprimée | Volume négligeable, pas de modalité sémantique évidente |
| Vitesses > 130 km/h | NaN (imputation aval) | Seuil réglementaire max en France |
| `Non renseigné` | → `Inconnu` | Signal prédictif potentiel, conservé comme modalité |
| NaN sur Météo, Vitesse_num, Voies_num | Conservés | Imputation déléguée au pipeline de chaque binôme |
| Département → Région | 107 → 14 modalités | Évite l'explosion dimensionnelle en OHE |
| Encoding features (arbres) | `OrdinalEncoder` + ordre sémantique | Facilite l'interprétation des splits |
| Encoding features (linéaire) | `pd.get_dummies` | Évite d'imposer un ordre arbitraire |
| Encoding cible | Mapping manuel | Préserve l'ordre de gravité |
| Mappings | Sauvés en JSON | Indispensable pour interpréter les feature importances |
